In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("/code/src/")

In [ ]:
import time
import numpy as np
from numpy.typing import NDArray
import viser
from plyfile import PlyData
from typing import List, Optional, Dict
import json
import cv2

In [ ]:
from drone_core import get_poses_from_data, get_images_from_data
from geometry import R_to_quat, homo_pose_to_quat

## Helper functions

In [ ]:
def get_traj_camera_centers_pairs(scene_traj_data, traj_name, step=5):
    frames = scene_traj_data[traj_name]["frames"]

    camera_centers_measured = []
    camera_centers_colmap = []
    for i in range(0, len(frames), step):
        frame = frames[i]
        if "colmap_pose_c2w" not in frame:
            continue
        else:
            camera_centers_measured.append(np.array(frame["measured_pose_c2w"])[:3, 3])
            camera_centers_colmap.append(np.array(frame["colmap_pose_c2w"])[:3, 3])
    
    return camera_centers_measured, camera_centers_colmap

In [ ]:
def get_traj_frames_data(scene_traj_data:List, trajectory_name:str, 
                         cam_intrinsics_type:str="camera_intrinsic_colmap", 
                         c2w_pose_type:str="colmap_pose_c2w"):
    traj_data = scene_traj_data[trajectory_name]
    cam_intrinsics = traj_data[cam_intrinsics_type]

    frames = traj_data["frames"]

    loaded_frames = []

    for frame in frames:
        if c2w_pose_type not in frame:
            continue

        loaded_frame = {"file_name":frame["file_name"],
                        "pose_c2w":frame[c2w_pose_type],
                        "intrinsics":cam_intrinsics}
        loaded_frames.append(loaded_frame)

    return loaded_frames

## Classes

In [ ]:
def drone_DS_2_viser_pose(c2w):
    openGL_2_openCV_T = np.array([[1, 0, 0, 0],
                                  [0, -1, 0, 0],
                                  [0, 0, -1, 0],
                                  [0, 0, 0, 1]])
    c2w = c2w @ openGL_2_openCV_T
    xyzw, position = homo_pose_to_quat(c2w)

    wxyz = np.array([xyzw[3], xyzw[0], xyzw[1], xyzw[2]])
    position = np.array(position)

    return wxyz, position

In [ ]:
class ViserVisualization():
    def __init__(self, port):
        self.server = viser.ViserServer(port=port)

    def get_server(self):
        return self.server
    
    def add_pointcloud(self, pointcloud_file, scene_object_name, 
                       point_size=0.002, point_shape='circle',
                       color=[0, 0, 0]):
        ply_data = PlyData.read(pointcloud_file)
        vertices = ply_data["vertex"]
        points = np.vstack([vertices["x"],
                            vertices["y"],
                            vertices["z"]]).T.astype(np.float32)
        
        if "red" in vertices and "green" in vertices and "blue" in vertices:
            colors = np.vstack([[vertices["red"]],
                                vertices["green"],
                                vertices["blue"]]).T.astype(np.float32)
            colors /= 255
        else:
            # create a constant color with the given color value
            colors = np.full_like(points, color)
    
        pointcloud_handle = self.server.scene.add_point_cloud(
            name=scene_object_name,
            points=points,
            colors=colors,
            point_size=point_size,
            point_shape=point_shape
        )

        return pointcloud_handle
    
    def add_camera_frustums(self, poses_c2w:List[NDArray], images:Optional[List[NDArray]]=None,
                            names:Optional[List[str]]=None, H:int=600, W:int=900, 
                            v_fov_degree=70, scale=0.15, line_width=1.0, color=[0, 0, 0],
                            variant='wireframe', visible=True, image_downsample=5):
        if images is None:
            # create None list
            images = [None] * len('wireframe')
        if names is None:
            # create fake names
            names = [f"cam_{i:05}"  for i in range(len(poses_c2w))]

        v_fov = np.deg2rad(v_fov_degree)

        frustums = []

        for c2w, image, name in zip(poses_c2w, images, names):

            wxyz, position= drone_DS_2_viser_pose(c2w.copy())

            image = image[::image_downsample, ::image_downsample]

            frustum = self.server.scene.add_camera_frustum(
                            name = name,
                            fov = v_fov,
                            aspect=W/H,
                            scale = scale,
                            wxyz = wxyz,
                            position=position,
                            image=image,
                            line_width=line_width,
                            color = color,
                            variant = variant,
                            visible=visible
                        )

            frustums.append(frustum)
        
        return frustums
            

In [ ]:
class SceneVisualization():
    def __init__(self, dataset_root, scene_name, port):
        self.dataset_root = dataset_root
        self.scene_name = scene_name
        self.scene_dir = f"{self.dataset_root}/{self.scene_name}"
        self.viser_visualization = ViserVisualization(port)
        # set the correct coordinate system (NED)
        self.viser_visualization.get_server().scene.set_up_direction('-z')

        # set the scene paths
        scene_data_json = f"{self.scene_dir}/scene_data.json"
        self.scene_point_cloud_file = f"{self.scene_dir}/sparse_model.ply"
        # we will load the scene data
        with open(scene_data_json, "r") as f:
            self.scene_data = json.load(f)
        
        # create the visualization dict
        self.scene_vis_dict = {"trajectories":{}}
        for traj_name in self.scene_data["trajectories"].keys():
             self.scene_vis_dict["trajectories"][traj_name] = {"color":None,
                                                               "frustum_handlers":{}}

    def visualize_point_cloud(self):
        pointcloud_handler = self.viser_visualization.add_pointcloud(self.scene_point_cloud_file, 
                                                                     scene_object_name="/pointcloud")
        
        # populate the scene dict
        self.scene_vis_dict["pointcloud"] = {"handler":pointcloud_handler}

    def add_point_cloud_gui(self):

        server = self.viser_visualization.get_server()

        gui_show_box = server.gui.add_checkbox(
            "Show point cloud",
            initial_value=True
        )

        gui_size_slider = server.gui.add_slider(
            "Point Size",
            min=0.001,
            max=0.1,
            step=0.001,
            initial_value=0.002
        )

        @gui_show_box.on_update
        def _(_event):
            self.scene_vis_dict["pointcloud"]["handler"].visible = gui_show_box.value

        @gui_size_slider.on_update
        def _(_event):
            self.scene_vis_dict["pointcloud"]["handler"].point_size = gui_size_slider.value

        
        # add the GUI handlers
        self.scene_vis_dict["pointcloud"]["gui_box_handler"] = gui_show_box
        self.scene_vis_dict["pointcloud"]["gui_slider_handler"] = gui_show_box

    def visualize_world_coordinate(self, axes_raduis=0.03, axes_length=0.7):
        world_coord_handle = self.viser_visualization.get_server().scene.add_frame(
            "/world_coordinates",
            wxyz=(1.0, 0.0, 0.0, 0.0),
            position=(0.0, 0.0, 0.0),
            axes_radius=axes_raduis,
            axes_length=axes_length
        )

        self.scene_vis_dict["world_coord"] = {"handler":world_coord_handle}
    
    def add_world_coordinate_gui(self):

        gui_show_w_coord = self.viser_visualization.get_server().gui.add_checkbox(
            "Show world coordinate",
            initial_value=True
        )

        @gui_show_w_coord.on_update
        def _(_event):
            self.scene_vis_dict["world_coord"]["handler"].visible = gui_show_w_coord.value

        # add the GUI handlers
        self.scene_vis_dict["world_coord"]["gui_box_handler"] = gui_show_w_coord

        
    def visualize_traj_camera_frustums(self, traj_name, pose_type="colmap_pose_c2w",
                                       variant='filled', intrinsic_type="camera_intrinsic_colmap",
                                       scale=0.15, line_width=2.0, visible=True, image_downsample=5):
        # read the trajectory color
        traj_color = self.scene_data["trajectories"][traj_name]["color_value"]
        traj_color = [color_comp /255 for color_comp in traj_color]

        # load the frames data
        frames_data = get_traj_frames_data(self.scene_data["trajectories"],
                                           trajectory_name=traj_name,
                                           cam_intrinsics_type=intrinsic_type,
                                           c2w_pose_type=pose_type)
        
        # get the camera params
        if len(frames_data) > 0:
            sample_camera = frames_data[0]["intrinsics"]
        else:
            raise ValueError(f"No frames found in traj {traj_name}")
        
        
        H = int(sample_camera["h"])
        W = int(sample_camera["w"])
        fy = float(sample_camera["fl_y"])
        v_fov = 2 * np.atan(H/(2*fy))
        v_fov_degree =np.rad2deg(v_fov)

        poses_c2w = get_poses_from_data(frames_data)
        images = get_images_from_data(frames_data, self.scene_dir)
        images = [cv2.cvtColor(image, cv2.COLOR_BGR2RGB) for image in images]

        if pose_type == "colmap_pose_c2w":
            pose_source = "COLMAP"
        elif pose_type == "measured_pose_c2w":
            pose_source = "Measured"
        else:
            pose_source = "Unkown"

        frame_names = [f"/{traj_name}/{pose_source}/{frame['file_name'].split('/')[-1]}" for frame in frames_data]

        frustums_handlers = self.viser_visualization.add_camera_frustums(poses_c2w=poses_c2w, images=images,
                                                                        names=frame_names, H=H, W=W, 
                                                                        v_fov_degree=v_fov_degree,
                                                                        scale=scale, line_width=line_width,
                                                                        color=traj_color, variant=variant,
                                                                        visible=visible, image_downsample=image_downsample)
        
        # populate the visualization dict
        self.scene_vis_dict["trajectories"][traj_name]["color"] = traj_color
        self.scene_vis_dict["trajectories"][traj_name]["frustum_handlers"][pose_type] = frustums_handlers

    
    def add_trajs_folder(self):
        trajs_folder = self.viser_visualization.get_server().gui.add_folder("Trajectories")
        self.scene_vis_dict["trajs_folder_handler"] = trajs_folder

    def add_frustums_gui(self, traj_name, visible=True):

        server = self.viser_visualization.get_server()

        with self.scene_vis_dict["trajs_folder_handler"]:
            traj_folder = server.gui.add_folder(traj_name, expand_by_default=False)

            with traj_folder:
                gui_frustum_optim_show = server.gui.add_checkbox(
                    f"Optimized",
                    initial_value=visible
                )
                gui_frustum_meas_show = server.gui.add_checkbox(
                    f"Measured",
                    initial_value=visible
                )

                gui_frustum_error_show = server.gui.add_checkbox(
                    f"Error",
                    initial_value=visible
                )
                

        @gui_frustum_optim_show.on_update
        def _(_event, traj_name=traj_name):
            frustums = self.scene_vis_dict["trajectories"][traj_name]["frustum_handlers"]["colmap_pose_c2w"]
            for frustum in frustums:
                frustum.visible = gui_frustum_optim_show.value

        @gui_frustum_meas_show.on_update
        def _(_event, traj_name=traj_name):
            frustums = self.scene_vis_dict["trajectories"][traj_name]["frustum_handlers"]["measured_pose_c2w"]
            for frustum in frustums:
                frustum.visible = gui_frustum_meas_show.value

        self.scene_vis_dict["trajectories"][traj_name]["traj_folder"] = traj_folder

    def add_traj_details(self, traj_name):

        server = self.viser_visualization.get_server()
        traj_folder = self.scene_vis_dict["trajectories"][traj_name]["traj_folder"] 

        source_type = self.scene_data["trajectories"][traj_name]["source_type"]
        n_frames = self.scene_data["trajectories"][traj_name]["number_frames_in_traj"]
        n_missing_frames = self.scene_data["trajectories"][traj_name]["missing_colmap_frames"]

        average_rotation_error = self.scene_data["trajectories"][traj_name]["average_rot_error"]
        average_distance_error = self.scene_data["trajectories"][traj_name]["average_cam_center_error_distance"]
        
        with traj_folder:
            markdown = server.gui.add_markdown(
                content= f"""
            #### Details
            - **Source type**: {source_type}
            - **Number of frames**: {n_frames}
            - **Number of missing frames**: {n_missing_frames}
            #### Statistics
            - **Average rotation error**: {average_rotation_error:.2f}
            - **Average distance error**: {average_distance_error:.2f}
            """
            )
        divider = server.gui.add_divider()
    
    def add_camera_info_markdown(self):
        selected_info = self.viser_visualization.get_server().gui.add_markdown("""
        ### Selected Camera

        Click a camera frustum to inspect it.
        """)

        self.scene_vis_dict["selected_info"] = selected_info

    def make_click_callback(
        self,
        frustum_handle,
        traj_name,
        pose_source,
        frame_id,
        image_name,
        translation_error=None,
        rotation_error=None,
    ):
        @frustum_handle.on_click
        def _(_event):
            err_t = "Pose missing" if translation_error is None else f"{translation_error:.3f} m"
            err_r = "Pose missing" if rotation_error is None else f"{rotation_error:.2f}°"

            self.scene_vis_dict["selected_info"].content = f"""
    ### Selected Camera

    | Field | Value |
    |---|---|
    | Trajectory | `{traj_name}` |
    | Pose source | `{pose_source}` |
    | Frame | `{frame_id}` |
    | Image | `{image_name}` |
    | Translation error | `{err_t}` |
    | Rotation error | `{err_r}` |
    """
            
    def create_traj_click_callbacks(self, traj_name, pose_type="colmap_pose_c2w"):
        
        frustums = self.scene_vis_dict["trajectories"][traj_name]["frustum_handlers"][pose_type]
        frames = self.scene_data["trajectories"][traj_name]["frames"]

        if pose_type == "colmap_pose_c2w":
            pose_source = "COLMAP"
        elif pose_type == "measured_pose_c2w":
            pose_source = "Measured"
        else:
            pose_source = "Unkown"

        for i, frustum in enumerate(frustums):
            frame = frames[i]
            if "camera_center_error_distance" in frame:
                trans_error = frame["camera_center_error_distance"]
            else:
                trans_error = None
            
            if "rot_error" in frame:
                rot_error = frame["rot_error"]
            else:
                rot_error = None

            image_name = frame["file_name"].split("/")[-1]

            self.make_click_callback(frustum, traj_name=traj_name,
                                     pose_source=pose_source, frame_id=f"{i:05}",
                                     image_name=image_name, translation_error=trans_error,
                                     rotation_error=rot_error)

    

## Definitions

In [ ]:
src_dir = "/workspace/"
dataset_root = f"{src_dir}/datasets/processed"
scene_name = "backyard_2"

port = 8080

In [ ]:
scene_vis = SceneVisualization(dataset_root=dataset_root, scene_name=scene_name, port=port)

In [ ]:
scene_vis.visualize_world_coordinate()
scene_vis.add_world_coordinate_gui()
scene_vis.visualize_point_cloud()
scene_vis.add_point_cloud_gui()

In [ ]:
divider = scene_vis.viser_visualization.get_server().gui.add_divider()
scene_vis.add_camera_info_markdown()
divider = scene_vis.viser_visualization.get_server().gui.add_divider()
scene_vis.add_trajs_folder()

In [ ]:
trajectories = ["traj_1", "traj_2", "traj_3", "traj_4", 
                "traj_5", "traj_6", "traj_7", "traj_8",
                "traj_9", "traj_10", "traj_11", "traj_12",
                "traj_13", "traj_14", "traj_15", "traj_16",
                "traj_17", "traj_18", "traj_19", "traj_20"]

In [ ]:
for traj in trajectories:
    scene_vis.visualize_traj_camera_frustums(traj_name=traj, visible=False, image_downsample=7)
    scene_vis.visualize_traj_camera_frustums(traj_name=traj,pose_type="measured_pose_c2w", 
                                             variant='wireframe', visible=False, image_downsample=7)
    scene_vis.add_frustums_gui(traj, visible=False)
    scene_vis.add_traj_details(traj)
    scene_vis.create_traj_click_callbacks(traj)
    scene_vis.create_traj_click_callbacks(traj, pose_type="measured_pose_c2w")
    time.sleep(10)